## استدعاء المكتبات اللازمة


In [ ]:

import pandas as pd
import numpy as np

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

## 1. قراءة البيانات واستكشافها


In [2]:
#https://www.kaggle.com/datasets/aakash50897/churn-modellingcsv
df = pd.read_csv('/datasets/Churn_Modelling.csv')

print(df.columns)
print(df.info())
print(df['Exited'].value_counts())


Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null 

## 2. تحديد المدخلات والمتغير المستهدف


In [ ]:
X = df.drop(
    columns=['RowNumber', 'CustomerId', 'Surname', 'Exited']
)
Y = df['Exited']

## 3. تقسيم البيانات قبل المعالجة


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.25,
    random_state=42,
    stratify=Y
)


## 4. تحويل الأعمدة النصية إلى أرقام


In [ ]:
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(handle_unknown='ignore'),
         ['Geography', 'Gender'])
    ],
    remainder='passthrough'
)

## 5. إنشاء نموذج KNN مع خطوات المعالجة


In [ ]:
model = Pipeline(steps=[
    ('encoder', ct),
    ('scaler', StandardScaler()),
    ('oversampler', RandomOverSampler(random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])


## 6. تدريب النموذج


In [ ]:
model.fit(x_train, y_train)

## 7. التنبؤ باستخدام بيانات الاختبار


In [ ]:
y_pred = model.predict(x_test)

## 8. تقييم النموذج


In [2]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.714

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.73      0.80      1991
           1       0.38      0.66      0.49       509

    accuracy                           0.71      2500
   macro avg       0.64      0.69      0.64      2500
weighted avg       0.79      0.71      0.74      2500


Confusion Matrix:
[[1448  543]
 [ 172  337]]


## 9. تجربة قيم مختلفة من K


In [2]:
for k in [3, 5, 7, 9, 11]:
    model.set_params(knn__n_neighbors=k)
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)

    print(
        f"K = {k}, Accuracy = "
        f"{accuracy_score(y_test, y_pred):.4f}"
    )

K = 3, Accuracy = 0.7476
K = 5, Accuracy = 0.7140
K = 7, Accuracy = 0.7204
K = 9, Accuracy = 0.7260
K = 11, Accuracy = 0.7272
